# Import Library

In [ ]:
#Initial setup
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display

import pandas as pd
import os
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from decimal import Decimal, ROUND_HALF_UP
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.linear_model import LogisticRegression
import urllib.request
import joblib

# Import dataset

In [ ]:
fileDir = "./Data/"
fileName = "pro_data_common_val_Assay.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_data = temp
display(pro_data)

In [ ]:
fileDir = "./Data/"
fileName = "pro_data_common_dis_Assay.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_data_dis = temp
display(pro_data_dis)

In [ ]:
fileDir = "./Data/"
fileName = "pro_sample_val.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_sample = temp
display(pro_sample)

In [ ]:
fileDir = "./Data/"
fileName = "map_OID_Uni_Sym.csv"

path = os.path.join(fileDir, fileName)
IDdata = pd.read_csv(path)
IDdata.set_index(IDdata.columns[0],inplace=True)
IDdata.index.name = None
display(IDdata)

In [ ]:
fileDir = "./Data/"
fileName = "DEP_list.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
display(temp)
DEPlist = temp

In [ ]:
fileDir = "./Data/"
fileName = "pro_data_common_dis.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_data_dis= temp
display(pro_data_dis)

In [ ]:
fileDir = "../Proteomics/Data/"
fileName = "CA199.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
temp = temp.sort_values('SampleID')
temp = temp.set_index('SampleID')
ca199_dis= temp
display(ca199_dis)

In [ ]:
fileDir = "./Data/"
fileName = "pro_sample_dis.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_sample_dis = temp
display(pro_sample_dis)

log変換、欠損値の処理、標準化済み

## CA-19-9とIL2RAのデータ読み込み

In [ ]:
ca199 = pro_sample[['CA19_9']].copy()
display(ca199)

il2ra = pro_sample[['sIL2R']].copy()
display(il2ra)

In [ ]:
print((ca199['CA19_9'] <= 0).sum())
print((il2ra['sIL2R'] <= 0).sum())

In [ ]:
ca199['logvalue'] = np.log10(ca199['CA19_9'])
il2ra['logvalue'] = np.log10(il2ra['sIL2R'])


# CA-19-9の値を予想するモデル作成

## IL2RAとプロテオミクスデータを結合したものを作成

In [ ]:
display(pro_data)
display(ca199)
display(il2ra)

In [ ]:
temp = pro_data.copy()
temp["IL2RA"] = il2ra["logvalue"]
display(temp["IL2RA"])
pro_data_il2ra = temp


## LASSO回帰

## ランダムフォレスト

### 全プロテオミクス

In [ ]:
X = pro_data_il2ra
y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=123)
r2_list = []
mae_list = []
# fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(kf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 3]
    }
    
    model = RandomForestRegressor(
        criterion='absolute_error',
        random_state=123,
        n_jobs=-1 
    )
    
    cv_gen = KFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='r2',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    model.fit(X_train, y_train.values.ravel())

    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    r2_list.append(r2)
    mae_list.append(mae)
    

    print(f"Fold {i+1} - Test R2 Score: {r2:.3f} | Test MAE: {mae:.3f}")

print("\n=== 全体平均結果 ===")
print(f"平均R2スコア: {np.mean(r2_list):.3f}")
print(f"平均MAE: {np.mean(mae_list):.3f}")



## 予測されたCA-19-9の値を見る

In [ ]:
display(ca199)

### 全プロテオミクス

In [ ]:
X = pro_data
y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=123
)

tempI = X_test.index

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=123, n_jobs=-1))
])

pipeline.fit(X_train, y_train.values.ravel())


y_pred = pipeline.predict(X_test)

comparison_df = pd.DataFrame({
    'Actual_CA19_9': np.array(y_test).flatten(),
    'Predicted_CA19_9': np.array(y_pred).flatten()
}, index=tempI)
comparison_df['Abs_Diff'] = np.abs(comparison_df['Actual_CA19_9'] - comparison_df['Predicted_CA19_9'])
comparison_df['CancerStatus'] = pro_sample.loc[tempI, 'CaCo_Status']
comparison_df = comparison_df.sort_values(by='Abs_Diff', ascending=False)


# 上位50サンプルを表示
print("--- 各サンプルの実際のCA19-9 vs 予測されたCA19-9 ---")
print(comparison_df.head(50))

plt.figure(figsize=(6, 6))

# Status（0と1）でグループ分けしてプロット
colors = {0: 'teal', 1: 'orange'}
labels = {0: 'Status 0 (Healthy)', 1: 'Status 1 (Cancer)'}

for status in [0, 1]:
    subset = comparison_df[comparison_df['CancerStatus'] == status]
    plt.scatter(subset['Actual_CA19_9'], subset['Predicted_CA19_9'], 
                alpha=0.7, c=colors[status], label=labels[status])

# 完璧に予測できた場合の補助線（既存の処理）
max_val = max(comparison_df['Actual_CA19_9'].max(), comparison_df['Predicted_CA19_9'].max())
min_val = min(comparison_df['Actual_CA19_9'].min(), comparison_df['Predicted_CA19_9'].min())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect Prediction')

plt.title("ALL Proteome : Actual vs Predicted CA19-9 by Status")
plt.xlabel("Actual CA19-9")
plt.ylabel("Predicted CA19-9")
plt.legend()
plt.grid(True)
plt.show()

# 1. パイプラインからLassoの係数を取り出す
coef = pipeline.named_steps['lasso'].coef_
feature_names = X.columns

# 2. データフレームにまとめて、影響度順にソート
importance_df = pd.DataFrame({
    'Protein': feature_names,
    'Coefficient': coef
}).assign(Abs_Coef = lambda x: x['Coefficient'].abs()) \
  .sort_values(by='Abs_Coef', ascending=False)

# 3. 係数が0でない（選ばれた）タンパク質のみ表示
selected_proteins = importance_df[importance_df['Coefficient'] != 0]
print(len(selected_proteins))
display(selected_proteins)

In [ ]:
from scipy.stats import pearsonr, spearmanr

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=123
)

tempI = X_test.index

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=123, n_jobs=-1))
])

pipeline.fit(X_train, y_train.values.ravel())


y_pred = pipeline.predict(X_test)

comparison_df = pd.DataFrame({
    'Actual_CA19_9': np.array(y_test).flatten(),
    'Predicted_CA19_9': np.array(y_pred).flatten()
}, index=tempI)
comparison_df['Abs_Diff'] = np.abs(comparison_df['Actual_CA19_9'] - comparison_df['Predicted_CA19_9'])
comparison_df['progress'] = pro_sample.loc[tempI, 'progress']
comparison_df = comparison_df.sort_values(by='Abs_Diff', ascending=False)

actual = np.asarray(y_test).ravel()
predicted = np.asarray(y_pred).ravel()

residual = predicted - actual

print("R2:", r2_score(actual, predicted))
print("RMSE:", np.sqrt(mean_squared_error(actual, predicted)))
print("MAE:", mean_absolute_error(actual, predicted))
print("Pearson r:", pearsonr(actual, predicted).statistic)
print("Spearman rho:", spearmanr(actual, predicted).statistic)

# 上位50サンプルを表示
print("--- 各サンプルの実際のCA19-9 vs 予測されたCA19-9 ---")
print(comparison_df.head(50))

plt.figure(figsize=(6, 6))

# Status（0と1）でグループ分けしてプロット
colors = {
    0:   '#009E73',  # Healthy：青緑
    400: '#FDE725',  # 黄色
    410: '#FCA636',  # オレンジ
    420: '#F66B19',  # 橙赤
    430: '#D73027',  # 赤
    440: '#7F0000'   # 濃い赤
}
labels = {0: 'Status 0 (Healthy)', 400: 'Status 1 (Cancer)',
          410: 'Status 2 (Cancer)',420: 'Status 3 (Cancer)',
          430: 'Status 4 (Cancer)',440: 'Status 5 (Cancer)'}

for status in [0 ,400,410,420,430,440]:
    subset = comparison_df[comparison_df['progress'] == status]
    plt.scatter(subset['Actual_CA19_9'], subset['Predicted_CA19_9'], 
                alpha=0.7, c=colors[status], label=labels[status])

# 完璧に予測できた場合の補助線（既存の処理）
max_val = max(comparison_df['Actual_CA19_9'].max(), comparison_df['Predicted_CA19_9'].max())
min_val = min(comparison_df['Actual_CA19_9'].min(), comparison_df['Predicted_CA19_9'].min())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect Prediction')

plt.title("ALL Proteome : Actual vs Predicted CA19-9 by Status")
plt.xlabel("Actual CA19-9")
plt.ylabel("Predicted CA19-9")
plt.legend()
plt.grid(True)
plt.show()

# 1. パイプラインからLassoの係数を取り出す
coef = pipeline.named_steps['lasso'].coef_
feature_names = X.columns

# 2. データフレームにまとめて、影響度順にソート
importance_df = pd.DataFrame({
    'Protein': feature_names,
    'Coefficient': coef
}).assign(Abs_Coef = lambda x: x['Coefficient'].abs()) \
  .sort_values(by='Abs_Coef', ascending=False)

# 3. 係数が0でない（選ばれた）タンパク質のみ表示
selected_proteins = importance_df[importance_df['Coefficient'] != 0]
print(len(selected_proteins))
display(selected_proteins)

In [ ]:
# selected proteinの詳細を見るためエクスポート
selected_proteins.to_csv("./Data/lasso_selectedProteins.csv", index=False)

In [ ]:


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=123
)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# 1. パイプラインの定義 (標準化 + LassoCV)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=123, n_jobs=-1))
])

# 2. 交差検証の実行 (R^2スコアを算出)
# yは.values.ravel()で1次元配列に変換します
scores = cross_val_score(pipeline, X, y.values.ravel(), cv=5, scoring='r2')

print(f"各分割のR^2スコア: {scores}")
print(f"平均R^2: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")


### DEPlistのみ

In [ ]:
cols = list(DEPlist.index)
temp = pro_data.loc[:, pro_data.columns.isin(cols)].copy()

X = pd.concat(
    [temp ],
    axis=1
)
y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
from scipy.stats import pearsonr, spearmanr

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=123
)

tempI = X_test.index

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=123, n_jobs=-1))
])

pipeline.fit(X_train, y_train.values.ravel())


y_pred = pipeline.predict(X_test)

comparison_df = pd.DataFrame({
    'Actual_CA19_9': np.array(y_test).flatten(),
    'Predicted_CA19_9': np.array(y_pred).flatten()
}, index=tempI)
comparison_df['Abs_Diff'] = np.abs(comparison_df['Actual_CA19_9'] - comparison_df['Predicted_CA19_9'])
comparison_df['progress'] = pro_sample.loc[tempI, 'progress']
comparison_df = comparison_df.sort_values(by='Abs_Diff', ascending=False)

actual = np.asarray(y_test).ravel()
predicted = np.asarray(y_pred).ravel()

residual = predicted - actual

print("R2:", r2_score(actual, predicted))
print("RMSE:", np.sqrt(mean_squared_error(actual, predicted)))
print("MAE:", mean_absolute_error(actual, predicted))
print("Pearson r:", pearsonr(actual, predicted).statistic)
print("Spearman rho:", spearmanr(actual, predicted).statistic)

# 上位50サンプルを表示
print("--- 各サンプルの実際のCA19-9 vs 予測されたCA19-9 ---")
print(comparison_df.head(50))

plt.figure(figsize=(6, 6))

# Status（0と1）でグループ分けしてプロット
colors = {
    0:   '#009E73',  # Healthy：青緑
    400: '#FDE725',  # 黄色
    410: '#FCA636',  # オレンジ
    420: '#F66B19',  # 橙赤
    430: '#D73027',  # 赤
    440: '#7F0000'   # 濃い赤
}
labels = {0: 'Status 0 (Healthy)', 400: 'Status 1 (Cancer)',
          410: 'Status 2 (Cancer)',420: 'Status 3 (Cancer)',
          430: 'Status 4 (Cancer)',440: 'Status 5 (Cancer)'}

for status in [0 ,400,410,420,430,440]:
    subset = comparison_df[comparison_df['progress'] == status]
    plt.scatter(subset['Actual_CA19_9'], subset['Predicted_CA19_9'], 
                alpha=0.7, c=colors[status], label=labels[status])

# 完璧に予測できた場合の補助線（既存の処理）
max_val = max(comparison_df['Actual_CA19_9'].max(), comparison_df['Predicted_CA19_9'].max())
min_val = min(comparison_df['Actual_CA19_9'].min(), comparison_df['Predicted_CA19_9'].min())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect Prediction')

plt.title("DEPlist + IL2RA : Actual vs Predicted CA19-9 by Status")
plt.xlabel("Actual CA19-9")
plt.ylabel("Predicted CA19-9")
plt.legend()
plt.grid(True)
plt.show()

# 1. パイプラインからLassoの係数を取り出す
coef = pipeline.named_steps['lasso'].coef_
feature_names = X.columns

# 2. データフレームにまとめて、影響度順にソート
importance_df = pd.DataFrame({
    'Protein': feature_names,
    'Coefficient': coef
}).assign(Abs_Coef = lambda x: x['Coefficient'].abs()) \
  .sort_values(by='Abs_Coef', ascending=False)

# 3. 係数が0でない（選ばれた）タンパク質のみ表示
selected_proteins = importance_df[importance_df['Coefficient'] != 0]
print(len(selected_proteins))
display(selected_proteins)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=123
)

tempI = X_test.index

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=123, n_jobs=-1))
])

pipeline.fit(X_train, y_train.values.ravel())


y_pred = pipeline.predict(X_test)

comparison_df = pd.DataFrame({
    'Actual_CA19_9': np.array(y_test).flatten(),
    'Predicted_CA19_9': np.array(y_pred).flatten()
}, index=tempI)
comparison_df['Abs_Diff'] = np.abs(comparison_df['Actual_CA19_9'] - comparison_df['Predicted_CA19_9'])
comparison_df['CancerStatus'] = pro_sample.loc[tempI, 'CaCo_Status']
comparison_df = comparison_df.sort_values(by='Abs_Diff', ascending=False)


# 上位50サンプルを表示
print("--- 各サンプルの実際のCA19-9 vs 予測されたCA19-9 ---")
print(comparison_df.head(50))

plt.figure(figsize=(6, 6))

# Status（0と1）でグループ分けしてプロット
colors = {0: 'teal', 1: 'orange'}
labels = {0: 'Status 0 (Healthy)', 1: 'Status 1 (Cancer)'}

for status in [0, 1]:
    subset = comparison_df[comparison_df['CancerStatus'] == status]
    plt.scatter(subset['Actual_CA19_9'], subset['Predicted_CA19_9'], 
                alpha=0.7, c=colors[status], label=labels[status])

# 完璧に予測できた場合の補助線（既存の処理）
max_val = max(comparison_df['Actual_CA19_9'].max(), comparison_df['Predicted_CA19_9'].max())
min_val = min(comparison_df['Actual_CA19_9'].min(), comparison_df['Predicted_CA19_9'].min())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect Prediction')

plt.title("DEPlist+IL2RA : Actual vs Predicted CA19-9 by Status")
plt.xlabel("Actual CA19-9")
plt.ylabel("Predicted CA19-9")
plt.legend()
plt.grid(True)
plt.show()

# 1. パイプラインからLassoの係数を取り出す
coef = pipeline.named_steps['lasso'].coef_
feature_names = X.columns

# 2. データフレームにまとめて、影響度順にソート
importance_df = pd.DataFrame({
    'Protein': feature_names,
    'Coefficient': coef
}).assign(Abs_Coef = lambda x: x['Coefficient'].abs()) \
  .sort_values(by='Abs_Coef', ascending=False)

# 3. 係数が0でない（選ばれた）タンパク質のみ表示
selected_proteins = importance_df[importance_df['Coefficient'] != 0]
print(len(selected_proteins))
display(selected_proteins)

# 群にわける

In [ ]:
X = pro_data
y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
from scipy.stats import pearsonr, spearmanr

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=123
)

tempI = X_test.index

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=123, n_jobs=-1))
])

pipeline.fit(X_train, y_train.values.ravel())


y_pred = pipeline.predict(X_test)

comparison_df = pd.DataFrame({
    'Actual_CA19_9': np.array(y_test).flatten(),
    'Predicted_CA19_9': np.array(y_pred).flatten()
}, index=tempI)
comparison_df['Abs_Diff'] = np.abs(comparison_df['Actual_CA19_9'] - comparison_df['Predicted_CA19_9'])
comparison_df['progress'] = pro_sample.loc[tempI, 'progress']
comparison_df = comparison_df.sort_values(by='Abs_Diff', ascending=False)

actual = np.asarray(y_test).ravel()
predicted = np.asarray(y_pred).ravel()

residual = predicted - actual

print("R2:", r2_score(actual, predicted))
print("RMSE:", np.sqrt(mean_squared_error(actual, predicted)))
print("MAE:", mean_absolute_error(actual, predicted))
print("Pearson r:", pearsonr(actual, predicted).statistic)
print("Spearman rho:", spearmanr(actual, predicted).statistic)

# # 上位50サンプルを表示
# print("--- 各サンプルの実際のCA19-9 vs 予測されたCA19-9 ---")
# print(comparison_df.head(50))

plt.figure(figsize=(6, 6))

# Status（0と1）でグループ分けしてプロット
colors = {
    0:   '#009E73',  # Healthy：青緑
    400: '#FDE725',  # 黄色
    410: '#FCA636',  # オレンジ
    420: '#F66B19',  # 橙赤
    430: '#D73027',  # 赤
    440: '#7F0000'   # 濃い赤
}
labels = {0: 'Status 0 (Healthy)', 400: 'Status 1 (Cancer)',
          410: 'Status 2 (Cancer)',420: 'Status 3 (Cancer)',
          430: 'Status 4 (Cancer)',440: 'Status 5 (Cancer)'}

for status in [0 ,400,410,420,430,440]:
    subset = comparison_df[comparison_df['progress'] == status]
    plt.scatter(subset['Actual_CA19_9'], subset['Predicted_CA19_9'], 
                alpha=0.7, c=colors[status], label=labels[status])

# 完璧に予測できた場合の補助線（既存の処理）
max_val = max(comparison_df['Actual_CA19_9'].max(), comparison_df['Predicted_CA19_9'].max())
min_val = min(comparison_df['Actual_CA19_9'].min(), comparison_df['Predicted_CA19_9'].min())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect Prediction')

plt.title("ALL Proteome : Actual vs Predicted CA19-9 by Status")
plt.xlabel("Actual CA19-9")
plt.ylabel("Predicted CA19-9")
plt.legend()
plt.grid(True)
plt.show()

# 1. パイプラインからLassoの係数を取り出す
coef = pipeline.named_steps['lasso'].coef_
feature_names = X.columns

# 2. データフレームにまとめて、影響度順にソート
importance_df = pd.DataFrame({
    'Protein': feature_names,
    'Coefficient': coef
}).assign(Abs_Coef = lambda x: x['Coefficient'].abs()) \
  .sort_values(by='Abs_Coef', ascending=False)

# # 3. 係数が0でない（選ばれた）タンパク質のみ表示
# selected_proteins = importance_df[importance_df['Coefficient'] != 0]
# print(len(selected_proteins))
# display(selected_proteins)

In [ ]:
display(comparison_df)

In [ ]:
import numpy as np
import pandas as pd

# 1. 予測値 - 実測値を計算
comparison_df["Difference"] = (
    comparison_df["Predicted_CA19_9"]
    - comparison_df["Actual_CA19_9"]
)

# 2. 差の平均と標準偏差
diff_mean = comparison_df["Difference"].mean()
diff_sd = comparison_df["Difference"].std()

print(f"差の平均: {diff_mean:.4f}")
print(f"差のSD: {diff_sd:.4f}")

# 3. 平均 + 1 SD以上、平均 - 1 SD以下を抽出
upper_group = comparison_df[
    comparison_df["Difference"] >= diff_mean + diff_sd
].copy()

lower_group = comparison_df[
    comparison_df["Difference"] <= diff_mean - diff_sd
].copy()

print(f"+1 SD以上のサンプル数: {len(upper_group)}")
print(f"-1 SD以下のサンプル数: {len(lower_group)}")

# 4. 結果を確認
display(upper_group.sort_values("Difference", ascending=False))
display(lower_group.sort_values("Difference", ascending=True))

In [ ]:
import numpy as np
import pandas as pd

# 1. 予測値 - 実測値を計算
comparison_df["Difference"] = (
    comparison_df["Predicted_CA19_9"]
    - comparison_df["Actual_CA19_9"]
)

# 2. 差の平均と標準偏差
diff_mean = comparison_df["Difference"].mean()
diff_sd = comparison_df["Difference"].std()

print(f"差の平均: {diff_mean:.4f}")
print(f"差のSD: {diff_sd:.4f}")

# 3. + 1 SD以上、- 1 SD以下を抽出
upper_group = comparison_df[
    comparison_df["Difference"] >=  diff_sd
].copy()

lower_group = comparison_df[
    comparison_df["Difference"] <=  -diff_sd
].copy()

print(f"+1 SD以上のサンプル数: {len(upper_group)}")
print(f"-1 SD以下のサンプル数: {len(lower_group)}")

# 4. 結果を確認
display(upper_group.sort_values("Difference", ascending=False))
display(lower_group.sort_values("Difference", ascending=True))

In [ ]:
# サンプルIDと群情報を保存
group_df = pd.concat([
    pd.DataFrame({
        "SampleID": upper_group.index.astype(str),
        "Group": "Upper"
    }),
    pd.DataFrame({
        "SampleID": lower_group.index.astype(str),
        "Group": "Lower"
    })
], ignore_index=True)

group_df.to_csv(
    "./Data/CA19_9_difference_groups.csv",
    index=False,
    encoding="utf-8-sig"
)

print(group_df["Group"].value_counts())